In [1]:
import numpy as np 
import sys
import time
import csv
import os
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
import torch
from pathlib import Path
import hydra
from omegaconf import OmegaConf
from datetime import datetime

module_path = '/home/cpanourg/projects/2-hdvc/'

if module_path not in sys.path:
    sys.path.append(module_path)

from src.utils import read_fvecs, write_fvecs
from src.utils import append_or_create_csv


# Imports experiments (necessary to register experiments)
from lib.Qinco.qinco.qinco_tasks import QincoConvertTask, QincoEvalTask, QincoTrainTask
from lib.Qinco.qinco.search.search_tasks import (
    BuildIndexTask,
    EncodeDBTask,
    IVFTrainTask,
    SearchTask,
    TrainPairwiseDecoderTask,
)
# from lib.Qinco

/home/cpanourg/.conda/envs/dtwrl_env2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_fp = '/mnthdd/cpanourg/2-hdvc/' # thalia
# data_fp = '/data/cpanourg/2-hdvc/' # urania 

print("Loading GIST dataset...")

# Loading the GIST dataset
db = np.array(read_fvecs(f'{data_fp}data/gist/gist_base.fvecs'))
qr = np.array(read_fvecs(f'{data_fp}data/gist/gist_query.fvecs'))

train_ratio = 0.1 
n_samples = int(train_ratio * db.shape[0])

train_set_fp = f'{data_fp}data/gist/gist_train_{train_ratio}.fvecs'

train_set = db[np.random.choice(db.shape[0], size=n_samples, replace=False)]

print(f"Train set size: {n_samples}")

write_fvecs(train_set_fp, train_set)

Loading GIST dataset...
Reading File - /mnthdd/cpanourg/2-hdvc/data/gist/gist_base.fvecs:

(1000000, 960)
Reading File - /mnthdd/cpanourg/2-hdvc/data/gist/gist_query.fvecs:(1000, 960)
Train set size: 100000
Writing File - /mnthdd/cpanourg/2-hdvc/data/gist/gist_train_0.1.fvecs:(100000, 960)


In [3]:
cfg = OmegaConf.load("/home/cpanourg/projects/2-hdvc/lib/Qinco/config/qinco_cfg.yaml")


EXPERIMENTS = {
    "train": QincoTrainTask,
    "eval_valset": QincoTrainTask,
    "eval": QincoEvalTask,
    "eval_time": QincoEvalTask,
    "convert": QincoConvertTask,
    "ivf_centroids": IVFTrainTask,
    "encode": EncodeDBTask,
    "build_index": BuildIndexTask,
    "train_pairwise_decoder": TrainPairwiseDecoderTask,
    "search": SearchTask,
}


# Get current datetime
now = datetime.now()

# Format as string: year_month_day_hour_minute_second
datetime_str = now.strftime("%Y_%m_%d_%H_%M_%S")

print(datetime_str)


cfg.task = 'train'
cfg.output = f'/mnthdd/cpanourg/2-hdvc/results/qinco2/qinco_weights_{datetime_str}.pt'

cfg.L = 16
cfg.de = 384
cfg.dh = 384

cfg.A = 16
cfg.B = 32
cfg.M = 8
cfg.K = 256
cfg.ivf_K = 1048576

cfg.epochs = 1

cfg.db = '/mnthdd/cpanourg/2-hdvc/data/gist/gist_base.fvecs'
cfg.trainset = train_set_fp 

# Before running the experiment
cfg.epochs = 1
print(f"Checking epochs setting: {cfg.epochs}")  # Add this to verify the value

# You can also try setting it this way:
# OmegaConf.update(cfg, "epochs", 1, merge=True)

2025_10_06_10_46_17
Checking epochs setting: 1


In [4]:
expe = EXPERIMENTS[cfg.task](cfg)


Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [5]:
expe.accelerator.print(f"====================== RUNNING TASK {cfg.task}")
expe.run()
expe.accelerator.print("Task done")
expe.accelerator.end_training()  # Destroy process group



[T_total=00:00:37 | T_train=00:00:00 | T_inference=00:00:37] inference on validation split 10 / 10 [[MSE=1.89353]]
[T_total=00:05:27 | T_train=00:04:49 | T_epoch=00:04:49] train 88 / 88 (step 87) lr=0.000266667 loss=7.76917 (avg=7.77091) [[all losses: loss_substep=3.53268 ; mse_loss=4.23649 ; total_loss=7.76917]]
[T_total=00:05:59 | T_train=00:04:49 | T_inference=00:00:31] inference on validation split 10 / 10 [[MSE=0.771023]]
[T_total=00:10:49 | T_train=00:09:39 | T_epoch=00:04:49] train 88 / 88 (step 175) lr=0.000533333 loss=7.00025 (avg=6.23165) [[all losses: loss_substep=3.16951 ; mse_loss=3.83073 ; total_loss=7.00025]]
[T_total=00:11:20 | T_train=00:09:39 | T_inference=00:00:31] inference on validation split 10 / 10 [[MSE=0.692453]]


KeyboardInterrupt: 

In [ ]:
expe

: 